## Ансамбли деревьев

In [40]:
import pandas as pd
import optuna
import numpy as np

from sklearn.ensemble import BaggingRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import ElasticNet

### Задача регрессии

In [9]:
regression = pd.read_csv("../data/diamonds_filtered.csv")

In [11]:
y_regression = regression['price']
x_regression = regression.drop('price', axis=1)

In [21]:
results = []
def metrics(model, model_name, r2):
    x_train, x_test, y_train, y_test = train_test_split(
    x_regression, y_regression, test_size=0.2, random_state=81)

    model.fit(x_train, y_train)

    y_pred = model.predict(x_test)

    metrics = {
        'Model': model_name,
        'MAE': round(mean_absolute_error(y_test, y_pred), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
        'MAPE': round(mean_absolute_percentage_error(y_test, y_pred), 4),
        'R2 (Test)': round(model.score(x_test, y_test), 5),
        'R2 (CV)': round(r2, 5) 
    }

    results.append(metrics)

### BaggingRegressor

In [ ]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 5, 15),
        'max_samples': trial.suggest_float('max_samples', 0.5, 0.8),
        'max_features': trial.suggest_float('max_features', 0.5, 1.0),
        'bootstrap': True,
        'bootstrap_features': False,
        'n_jobs': -1,
        'random_state': 81,
        'verbose': 0
    }

    model = BaggingRegressor(**params)

    kf = KFold(n_splits=5, shuffle=True, random_state=81)
    score = cross_val_score(
        estimator=model, 
        X=x_regression, 
        y=y_regression, 
        scoring='r2', 
        cv=kf, 
        n_jobs=-1
    ).mean()

    return score

In [23]:
study = optuna.create_study(direction='maximize')
study.optimize(func=objective, n_trials=50, n_jobs=-1)

[I 2026-04-11 10:27:16,520] A new study created in memory with name: no-name-4742e1fb-4ac5-41be-abcc-25179bd20f3e
/Users/irinaaristova/Documents/Programming/ml-learning/venv/lib/python3.9/site-packages/sklearn/ensemble/_bagging.py:1315: UserWarning: Some inputs do not have OOB scores. This probably means too few estimators were used to compute any reliable oob estimates.
  warn(
/Users/irinaaristova/Documents/Programming/ml-learning/venv/lib/python3.9/site-packages/sklearn/ensemble/_bagging.py:1315: UserWarning: Some inputs do not have OOB scores. This probably means too few estimators were used to compute any reliable oob estimates.
  warn(
/Users/irinaaristova/Documents/Programming/ml-learning/venv/lib/python3.9/site-packages/sklearn/ensemble/_bagging.py:1315: UserWarning: Some inputs do not have OOB scores. This probably means too few estimators were used to compute any reliable oob estimates.
  warn(
/Users/irinaaristova/Documents/Programming/ml-learning/venv/lib/python3.9/site-pac

In [ ]:
print(f"Best params: {study.best_params}")
print(f"Best R^2: {study.best_value}")

metrics(BaggingRegressor(**study.best_params), 'BaggingRegressor', study.best_value)

Best params: {'n_estimators': 10, 'max_samples': 0.5676322199165681, 'max_features': 0.9467792663403839}
Best R^2: 0.9806168369014105


###  GradientBoostingRegressor

In [33]:
def objective(trial):
    params = {
        'loss': trial.suggest_categorical('loss', ['squared_error', 'absolute_error', 'huber', 'quantile']),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.5),
        'n_estimators': trial.suggest_int('n_estimators', 5, 500),
        'criterion': trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error']),
        'min_samples_split': trial.suggest_int('min_samples_split', 5, 30),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 5, 20),
        'max_depth': trial.suggest_int('max_depth', 2, 10),
        'min_impurity_decrease': trial.suggest_float('min_impurity_decrease', 0.0, 0.1),
        'random_state': 81,
        'ccp_alpha': trial.suggest_float('ccp_alpha', 0.0, 0.01)
    }

    model = GradientBoostingRegressor(**params)

    kf = KFold(n_splits=5, shuffle=True, random_state=81)
    score = cross_val_score(
        estimator=model, 
        X=x_regression, 
        y=y_regression, 
        scoring='r2', 
        cv=kf, 
        n_jobs=-1
    ).mean()

    return score

In [34]:
study = optuna.create_study(direction='maximize')
study.optimize(func=objective, n_trials=50)

[I 2026-04-11 10:53:15,352] A new study created in memory with name: no-name-3cee7410-64c5-46b0-80c0-18845379e2db


[I 2026-04-11 10:53:17,853] Trial 0 finished with value: 0.9814513873612001 and parameters: {'loss': 'squared_error', 'learning_rate': 0.468168629191333, 'n_estimators': 34, 'criterion': 'friedman_mse', 'min_samples_split': 10, 'min_samples_leaf': 11, 'max_depth': 7, 'min_impurity_decrease': 0.014582579336681901, 'ccp_alpha': 0.009170157580667205}. Best is trial 0 with value: 0.9814513873612001.
[I 2026-04-11 10:53:41,926] Trial 1 finished with value: 0.9449646336841724 and parameters: {'loss': 'absolute_error', 'learning_rate': 0.32635870597652417, 'n_estimators': 415, 'criterion': 'friedman_mse', 'min_samples_split': 26, 'min_samples_leaf': 5, 'max_depth': 10, 'min_impurity_decrease': 0.0799786427899379, 'ccp_alpha': 0.006250985814889285}. Best is trial 0 with value: 0.9814513873612001.
[I 2026-04-11 10:53:49,034] Trial 2 finished with value: 0.9819994605424771 and parameters: {'loss': 'squared_error', 'learning_rate': 0.18889195643851645, 'n_estimators': 117, 'criterion': 'friedman_

In [35]:
print(f"Best params: {study.best_params}")
print(f"Best R^2: {study.best_value}")

metrics(GradientBoostingRegressor(**study.best_params), 'GradientBoostingRegressor', study.best_value)

Best params: {'loss': 'huber', 'learning_rate': 0.049257176968997535, 'n_estimators': 499, 'criterion': 'friedman_mse', 'min_samples_split': 7, 'min_samples_leaf': 20, 'max_depth': 6, 'min_impurity_decrease': 0.06272164915822126, 'ccp_alpha': 0.00492786541927825}
Best R^2: 0.9836882729439541


In [36]:
results_df = pd.DataFrame(results)
results_df

,Model,MAE,RMSE,MAPE,R2 (Test),R2 (CV)
0,BaggingRegressor,282.8983,525.0476,0.0820,0.97838,0.98062
1,GradientBoostingRegressor,236.5839,454.5862,0.0708,0.98379,0.98369


### StackingRegressor

In [41]:
def objective(trial):
    dt_max_depth = trial.suggest_int('dt_max_depth', 3, 5)
    dt_min_samples = trial.suggest_int('dt_min_samples_split', 2, 20)
    
    knn_neighbors = trial.suggest_int('knn_neighbors', 3, 25)
    knn_weights = trial.suggest_categorical('knn_weights', ['uniform', 'distance'])

    en_alpha = trial.suggest_float('en_alpha', 0.001, 1.0, )
    en_l1 = trial.suggest_float('en_l1_ratio', 0, 1)

    estimators = [
        ('dt', DecisionTreeRegressor(max_depth=dt_max_depth, 
                                     min_samples_split=dt_min_samples, 
                                     random_state=81)),
        ('knn', KNeighborsRegressor(n_neighbors=knn_neighbors, 
                                    weights=knn_weights))
    ]
    
    final_mod = ElasticNet(alpha=en_alpha, l1_ratio=en_l1, random_state=81)

    model = StackingRegressor(
        estimators=estimators,
        final_estimator=final_mod,
        cv=5, 
        n_jobs=-1
    )

    kf = KFold(n_splits=5, shuffle=True, random_state=81)

    score = cross_val_score(
        model, 
        x_regression, 
        y_regression, 
        scoring='r2', 
        cv=kf, 
        n_jobs=-1
    ).mean()

    return score

In [46]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

[I 2026-04-11 14:39:13,739] A new study created in memory with name: no-name-2e0bf05f-313b-4249-89da-1306bf831479
/Users/irinaaristova/Documents/Programming/ml-learning/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/irinaaristova/Documents/Programming/ml-learning/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/irinaaristova/Documents/Programming/ml-learning/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/irinaaristova/Documents/Programming/ml-learning/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/irinaaristova/Documents/Programmin

In [48]:
bp = study.best_params

base_models = [
    ('dt', DecisionTreeRegressor(
        max_depth=bp['dt_max_depth'], 
        min_samples_split=bp['dt_min_samples_split'], 
        random_state=81
    )),
    ('knn', KNeighborsRegressor(
        n_neighbors=bp['knn_neighbors'], 
        weights=bp['knn_weights']
    ))
]

final_res = ElasticNet(
    alpha=bp['en_alpha'], 
    l1_ratio=bp['en_l1_ratio'], 
    random_state=81
)

best_stacking_model = StackingRegressor(
    estimators=base_models,
    final_estimator=final_res,
    cv=5,
    n_jobs=-1
)

metrics(best_stacking_model, 'StackingRegressor', study.best_value)

/Users/irinaaristova/Documents/Programming/ml-learning/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/irinaaristova/Documents/Programming/ml-learning/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/irinaaristova/Documents/Programming/ml-learning/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/irinaaristova/Documents/Programming/ml-learning/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/irinaaristova/Documents/Programming/ml-learning/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encount

In [49]:
results_df = pd.DataFrame(results)
results_df

,Model,MAE,RMSE,MAPE,R2 (Test),R2 (CV)
0,BaggingRegressor,282.8983,525.0476,0.0820,0.97838,0.98062
1,GradientBoostingRegressor,236.5839,454.5862,0.0708,0.98379,0.98369
2,StackingRegressor,459.8931,774.0993,0.1447,0.95300,0.93145


#### Вывод

- Наилучшую точность показал алгоритм `GradientBoostingRegressor`, достигнув самого высокого $R^2 = 0.984$ и потребовав меньше всего времени.
- `GradientBoostingRegressor` продемонстрировал высокую стабильность и сопоставимый результат $R^2 = 0.981$ на кросс-валидации, однако занял больше всего времени.
- `StackingRegressor` в данной конфигурации показал наименее эффективный результат с $R^2$ = 0.93. Это может быть связано с недостаточной сложностью базовых моделей или необходимостью более тонкой настройки мета-регрессора для эффективного комбинирования различных алгоритмов.

### Задача классификации

In [14]:
classification = pd.read_csv("../data/credit_card_fraud_filtered.csv")

In [15]:
y_classification = classification['fraud']
x_classification = classification.drop('fraud', axis=1)